In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import pystac_client
import planetary_computer as pc
import xarray as xr
from scipy.spatial import cKDTree

tqdm.pandas()

def random_date(start_date, end_date):
    """Generate a random date between start_date and end_date."""
    delta = end_date - start_date
    random_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=random_days)

def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]
    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )
    return ds

ds = load_terraclimate_dataset()
ds_pet = ds['pet'].sel(time=slice("2000-01-01", "2006-12-31"))

np.random.seed(42)  
num_samples = 200
lats = np.random.uniform(-90, 90, num_samples)
lons = np.random.uniform(-180, 180, num_samples)
start_date = datetime(2000, 1, 1)
end_date = datetime(2006, 12, 31)
dates = [random_date(start_date, end_date) for _ in range(num_samples)]

df = pd.DataFrame({
    'Latitude': lats,
    'Longitude': lons,
    'Sample Date': dates
})

def assign_nearest_pet(row, ds_pet):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'])
    # Select nearest
    nearest = ds_pet.sel(time=date, lat=lat, lon=lon, method='nearest')
    pet = float(nearest.values)
    return pet if not np.isnan(pet) else np.nan

df['pet'] = df.progress_apply(lambda row: assign_nearest_pet(row, ds_pet), axis=1)

df = df[['Longitude', 'Latitude', 'Sample Date', 'pet']]
df.to_csv('terraclimate_pet_2000_2006_200_samples.csv', index=False)

 16%|█▌        | 31/200 [30:42<2:47:25, 59.44s/it] 


ServiceResponseTimeoutError: Timeout on reading data from socket